In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


##Data Imported to Database

In [3]:
import pandas as pd
import mysql.connector
from getpass import getpass

# csv file path -> table name (order matters if you have foreign keys)
files = {
    r"C:\Users\hp\Desktop\pe\products_cleaned.csv":   "products",
    r"C:\Users\hp\Desktop\pe\cleaned_user_N.csv":   "users",
    r"C:\Users\hp\Desktop\pe\sessions.csv":    "sessions",
    r"C:\Users\hp\Desktop\pe\interactions_cleaned.csv":      "interactions",
    r"C:\Users\hp\Desktop\pe\purchases_cleaned.csv": "purchases",
    r"C:\Users\hp\Desktop\pe\reviews.csv": "reviews",
}

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password=getpass("MySQL password: "),
    database="E_Commerce_Customer_Intelligence"
)
cursor = conn.cursor()

for path, table in files.items():
    df = pd.read_csv(path)
    df = df.astype(object).where(pd.notnull(df), None)  # NaN -> NULL

    cols = ", ".join(f"`{c}`" for c in df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO `{table}` ({cols}) VALUES ({placeholders})"

    cursor.executemany(sql, df.values.tolist())
    conn.commit()
    print(f"{table}: {cursor.rowcount} rows inserted")

cursor.close()
conn.close()

MySQL password: ··········
products: 1000 rows inserted
users: 10000 rows inserted
sessions: 19315 rows inserted
interactions: 100000 rows inserted
purchases: 1737 rows inserted
reviews: 1253 rows inserted


##Data Fetching from database

In [5]:
import csv
import mysql.connector

# TODO: Replace 'YOUR_MYSQL_PASSWORD' with your actual local MySQL root password
conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Ak@shuniyal97",
    database="E_Commerce_Customer_Intelligence"
)

cursor = conn.cursor()

In [6]:
cursor = conn.cursor()

cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

interactions
products
purchases
reviews
sessions
users


In [7]:
import pandas as pd

In [13]:
q="""
  SELECT
    ROUND(COALESCE(SUM(total_amount), 0), 2)
        AS total_revenue,

    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS purchasing_customers,
    SUM(quantity) AS total_units_sold,
    ROUND(
        COALESCE(SUM(total_amount), 0)
        / NULLIF(COUNT(DISTINCT order_id), 0),
        2
    ) AS average_order_value
FROM purchases;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\1723189479.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,total_revenue,total_orders,purchasing_customers,total_units_sold,average_order_value
0,129510.85,1440,1287,2339.0,89.94


##Monthly Revenue

In [14]:
q="""
  SELECT
    DATE_FORMAT(order_date, '%Y-%m-01') AS sales_month,
    ROUND(SUM(total_amount), 2) AS revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_customers,
    SUM(quantity) AS units_sold
FROM purchases
GROUP BY DATE_FORMAT(order_date, '%Y-%m-01')
ORDER BY sales_month;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3401257369.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,sales_month,revenue,total_orders,unique_customers,units_sold
0,2023-01-01,197.24,3,3,6.0
1,2023-02-01,1124.23,11,11,22.0
2,2023-03-01,2706.91,32,28,49.0
3,2023-04-01,1918.38,23,21,42.0
4,2023-05-01,2552.58,24,23,44.0
5,2023-06-01,2549.52,26,24,41.0
6,2023-07-01,1904.17,23,23,33.0
7,2023-08-01,1487.89,16,15,28.0
8,2023-09-01,1469.43,22,22,35.0
9,2023-10-01,3587.70,24,23,44.0


##Daily sales trends

In [16]:
q="""
  SELECT
    DATE(order_date) AS sales_date,
    ROUND(SUM(total_amount), 2) AS daily_revenue,
    COUNT(DISTINCT order_id) AS orders,
    SUM(quantity) AS units_sold
FROM purchases
GROUP BY DATE(order_date)
ORDER BY sales_date;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\449052845.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,sales_date,daily_revenue,orders,units_sold
0,2023-01-15,11.48,1,2.0
1,2023-01-19,158.84,1,2.0
2,2023-01-31,26.92,1,2.0
3,2023-02-04,271.88,1,4.0
4,2023-02-06,121.90,1,1.0
...,...,...,...,...
825,2026-05-27,10.52,1,1.0
826,2026-05-28,205.78,2,5.0
827,2026-05-29,31.74,1,2.0
828,2026-05-30,289.62,1,3.0


##Revenue by year and quarter

In [18]:
q="""
  SELECT
    EXTRACT(YEAR FROM order_date) AS sales_year,
    EXTRACT(QUARTER FROM order_date) AS sales_quarter,
    ROUND(SUM(total_amount), 2) AS revenue,
    COUNT(DISTINCT order_id) AS orders
FROM purchases
GROUP BY
    EXTRACT(YEAR FROM order_date),
    EXTRACT(QUARTER FROM order_date)
ORDER BY sales_year, sales_quarter;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\2218361842.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,sales_year,sales_quarter,revenue,orders
0,2023,1,4028.38,46
1,2023,2,7020.48,73
2,2023,3,4861.49,61
3,2023,4,7700.45,69
4,2024,1,7728.25,101
5,2024,2,7356.04,74
6,2024,3,10723.95,105
7,2024,4,11295.66,133
8,2025,1,12376.89,147
9,2025,2,12236.03,143


##Find the top 20 customers by spending

In [20]:
q="""
  SELECT
    u.user_id,
    u.country,
    u.city,
    u.loyalty_tier,
    COUNT(DISTINCT p.order_id) AS total_orders,
    SUM(p.quantity) AS units_purchased,
    ROUND(SUM(p.total_amount), 2) AS total_spent
FROM users u
JOIN purchases p
    ON u.user_id = p.user_id
GROUP BY
    u.user_id, u.country, u.city, u.loyalty_tier
ORDER BY total_spent DESC
LIMIT 20;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\2035929743.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,user_id,country,city,loyalty_tier,total_orders,units_purchased,total_spent
0,6b888357-4601-4b33-8543-6ca73a7f7b04,US,Jacobsonshire,bronze,1,6.0,1486.54
1,1d915659-c9f5-4c24-947d-ac08255d6130,CA,Markport,silver,1,3.0,889.77
2,fa5cefb0-2f37-485c-80a3-7df4ecffd90b,FR,Paul-sur-Mer,bronze,1,2.0,885.30
3,84b90ed5-de66-4334-9106-77cab2fefe5d,US,Port Melissamouth,bronze,1,3.0,773.02
4,fd5f1634-99b8-4950-b0b0-09903e2a1ac9,KR,연천군,bronze,1,4.0,732.96
5,68037692-db83-40a6-a183-bcac6b43ef64,FR,Sainte Julesdan,bronze,1,2.0,667.35
6,b9e5bd2d-ad50-416f-ba4e-6f6e37111919,US,Petersonton,bronze,1,2.0,585.09
7,2ac29460-6138-4c29-bbf7-3e3b05cd514d,SG,Phillipsland,bronze,1,1.0,548.64
8,8c08b507-1f47-4e21-beb5-258709f5c6d3,US,Port Kathleen,silver,1,2.0,544.65
9,c7d4af0f-6f0b-45fd-b222-7316c071f512,US,North Danielleberg,bronze,1,1.0,541.90


##Identify repeat customers

In [23]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [24]:
q="""
  SELECT
    user_id,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(total_amount), 2) AS total_spent
FROM purchases
GROUP BY user_id
HAVING COUNT(DISTINCT order_id) > 1
ORDER BY order_count DESC, total_spent DESC;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\860013552.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,user_id,order_count,total_spent
0,a35e6564-b69d-4a62-86f7-c500e1c2422e,4,78.78
1,bf1ee143-eea7-4b6d-b3db-bd8fdf344256,3,498.49
2,f0437b46-0861-4f8a-a3cb-c7400e1cfe20,3,409.10
3,c20d4c44-5b48-4450-b06c-0a8eef6fae05,3,367.90
4,0f7a21f8-40fc-4d44-99cf-9eaf87ba344f,3,322.22
5,63d77def-8408-4c23-b7da-9b0d40922301,3,302.62
6,d7e8b18a-5ba0-47f1-ac7b-c9a65a60e095,2,489.59
7,8391e2bc-db17-477f-a417-b9ca66827a79,2,484.12
8,5aca5022-a81b-4c9b-af11-5e3f04fa7db9,2,470.77
9,8331e250-14fa-4372-99fc-ea646c6975e8,2,447.47


##Classify customers by purchase frequency

In [26]:
q="""
 WITH customer_orders AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(total_amount) AS total_spent
    FROM purchases
    GROUP BY user_id
)
SELECT
    CASE
        WHEN order_count = 1 THEN 'One-time customer'
        WHEN order_count BETWEEN 2 AND 4 THEN 'Occasional customer'
        WHEN order_count BETWEEN 5 AND 9 THEN 'Frequent customer'
        ELSE 'Very frequent customer'
    END AS customer_segment,
    COUNT(*) AS customers,
    ROUND(AVG(total_spent), 2) AS avg_customer_spend
FROM customer_orders
GROUP BY 1
ORDER BY customers DESC;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3102013749.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,customer_segment,customers,avg_customer_spend
0,One-time customer,1141,92.17
1,Occasional customer,146,166.74


##Best-selling products

In [28]:
q="""
 SELECT
    pr.product_id,
    pr.product_name,
    pr.category,
    pr.brand,
    SUM(p.quantity) AS units_sold,
    ROUND(SUM(p.total_amount), 2) AS revenue,
    COUNT(DISTINCT p.order_id) AS orders
FROM products pr
JOIN purchases p
    ON pr.product_id = p.product_id
GROUP BY
    pr.product_id, pr.product_name, pr.category, pr.brand
ORDER BY units_sold DESC
LIMIT 20;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\1185858838.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,product_id,product_name,category,brand,units_sold,revenue,orders
0,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,Apple Accessorie Stand,Electronics,Apple,88.0,6140.92,84
1,c1ab2206-9b50-4aaa-9446-f322d0976ea6,The Data Structures: A Comprehensive Guide,Books,Simon & Schuster,84.0,1669.20,43
2,dbe68f3f-99a8-4ef2-a103-908a6fca244d,Wilson Fitness Equipment Pro,Sports & Outdoors,Wilson,71.0,3471.85,55
3,26fa0037-c745-4789-8a44-9f3dfb7f25f7,General Mills Organic Snack,Grocery & Gourmet,General Mills,59.0,1087.82,28
4,5d07bcd4-3670-4a0d-91f6-e36d7d335597,Adidas Sterling Silver Jewelry,Clothing & Accessories,Adidas,56.0,1524.11,40
5,88f38eae-9124-4557-a285-248fc1cad30c,Nestle Pantry Staple Multipack,Grocery & Gourmet,Nestle,56.0,433.08,32
6,58fd8edb-60ef-4856-bf1f-644ffc133d12,Wilson Book19 Fitness Equipment,Sports & Outdoors,Wilson,55.0,5883.08,41
7,eaaf2b74-b09a-4bf5-b0b2-09021044cab8,Instant Pot Microfiber Kitchenware,Home & Kitchen,Instant Pot,55.0,5218.45,41
8,c2e39751-db17-424f-8d71-61ec7ecad424,Neutrogena Bath & Body Lotion,Beauty & Personal Care,Neutrogena,54.0,1818.69,37
9,d3701c21-709a-4090-bab2-020261ce8a92,Cuisinart Smart Appliance,Home & Kitchen,Cuisinart,53.0,2363.42,41


##Products generating the most revenue

In [30]:
q="""
SELECT
    pr.product_id,
    pr.product_name,
    pr.category,
    pr.brand,
    SUM(p.quantity) AS units_sold,
    ROUND(SUM(p.total_amount), 2) AS total_revenue
FROM products pr
JOIN purchases p
    ON pr.product_id = p.product_id
GROUP BY
    pr.product_id, pr.product_name, pr.category, pr.brand
ORDER BY total_revenue DESC
LIMIT 20;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\292026941.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,product_id,product_name,category,brand,units_sold,total_revenue
0,bbc99c6e-1552-4677-8a8f-61a2e541e6c8,Apple Book4 Ultrabook,Electronics,Apple,28.0,7202.93
1,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,Apple Accessorie Stand,Electronics,Apple,88.0,6140.92
2,58fd8edb-60ef-4856-bf1f-644ffc133d12,Wilson Book19 Fitness Equipment,Sports & Outdoors,Wilson,55.0,5883.08
3,eaaf2b74-b09a-4bf5-b0b2-09021044cab8,Instant Pot Microfiber Kitchenware,Home & Kitchen,Instant Pot,55.0,5218.45
4,dbe68f3f-99a8-4ef2-a103-908a6fca244d,Wilson Fitness Equipment Pro,Sports & Outdoors,Wilson,71.0,3471.85
5,f81e1fd5-a01b-4fbe-81d6-5dc5ab19b03c,Castrol Prime7 Electronic,Automotive,Castrol,23.0,2791.99
6,1660a6f2-7ef5-4f47-a4aa-e917f41e014e,Tommy Hilfiger Womenswear Dress,Clothing & Accessories,Tommy Hilfiger,44.0,2503.33
7,d3701c21-709a-4090-bab2-020261ce8a92,Cuisinart Smart Appliance,Home & Kitchen,Cuisinart,53.0,2363.42
8,4b9c83ae-69fe-4826-ac62-bae53913ea33,Apple Accessorie Stand,Electronics,Apple,8.0,2318.48
9,8b875a73-88df-44ff-966d-5649d7a14d31,Nike Womenswear in Olive,Clothing & Accessories,Nike,34.0,2170.90


##Revenue by category and subcategory

In [32]:
q="""
SELECT
    pr.category,
    pr.subcategory,
    COUNT(DISTINCT pr.product_id) AS products_sold,
    SUM(p.quantity) AS units_sold,
    COUNT(DISTINCT p.order_id) AS orders,
    ROUND(SUM(p.total_amount), 2) AS revenue
FROM products pr
JOIN purchases p
    ON pr.product_id = p.product_id
GROUP BY pr.category, pr.subcategory
ORDER BY revenue DESC;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\2967607581.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,category,subcategory,products_sold,units_sold,orders,revenue
0,Sports & Outdoors,Fitness Equipment,14,154.0,116,10909.04
1,Electronics,Laptops,14,51.0,50,10685.59
2,Electronics,Accessories,7,105.0,99,9188.91
3,Electronics,Smartphones,18,44.0,41,8824.83
4,Home & Kitchen,Kitchenware,13,114.0,85,7761.84
5,Clothing & Accessories,Womenswear,15,110.0,84,6005.94
6,Electronics,Headphones,11,26.0,25,4700.76
7,Sports & Outdoors,Camping Gear,10,65.0,44,4438.30
8,Clothing & Accessories,Menswear,22,98.0,65,4194.21
9,Automotive,Tools & Equipment,8,57.0,45,3893.62


##Find products with no sales

In [33]:
q="""
SELECT
    pr.product_id,
    pr.product_name,
    pr.category,
    pr.price,
    pr.stock_quantity,
    pr.date_added
FROM products pr
LEFT JOIN purchases p
    ON pr.product_id = p.product_id
WHERE p.purchase_id IS NULL
ORDER BY pr.date_added DESC;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3992669512.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,product_id,product_name,category,price,stock_quantity,date_added
0,5421da2a-4a0c-437d-934d-d2384fae9de4,The Meditation: The Definitive Guide,Books,8.37,1,2026-05-27
1,5d4ad04e-36bd-4988-a98d-d34720e44811,"LEGO Jack 27"" Figure",Toys & Games,8.21,3,2026-05-22
2,9256e0fe-5645-4196-a875-26b97940487e,Coleman Premium Fitness Equipment,Sports & Outdoors,37.82,2,2026-05-22
3,196e5f63-122c-4d20-abee-9b92d298b5e5,Nike Premium Womenswear,Clothing & Accessories,19.85,3,2026-05-19
4,4f9e7267-cd6f-42ec-916d-7bb0957a5f19,Apple Essential Sport Watch,Electronics,23.90,2,2026-05-11
5,85016e04-8cd0-47e3-8b77-52d6de00cfcf,Nike Premium Menswear,Clothing & Accessories,40.39,3,2026-05-05
6,c3f78b59-fd61-4c40-91f2-a5ae1e894b43,Wilson Outdoor Recreation Set,Sports & Outdoors,39.25,2,2026-05-03
7,2758ff6a-d2a3-43d0-ba31-3b52bebcbe2a,Olay Bath & Body Scrub,Beauty & Personal Care,18.27,1,2026-04-29
8,520cf0b3-8fe9-42ba-a245-f42e60a656d9,Wilson Edge13 Camping Gear,Sports & Outdoors,29.52,8,2026-04-29
9,776df9ea-e13e-40b6-bb01-fa51110f4390,A premium legacy,Books,20.53,2,2026-04-24


##Analyze customer engagement by product

In [35]:
q="""
SELECT
    pr.product_id,
    pr.product_name,
    pr.category,
    COUNT(i.interaction_id) AS total_interactions,
    COUNT(DISTINCT i.user_id) AS unique_users,
    ROUND(AVG(i.dwell_time_ms), 2) AS avg_dwell_time_ms
FROM products pr
JOIN interactions i
    ON pr.product_id = i.product_id
GROUP BY
    pr.product_id, pr.product_name, pr.category
ORDER BY total_interactions DESC
LIMIT 20;
"""
df=pd.read_sql(q,conn)
df

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3397766062.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(q,conn)


,product_id,product_name,category,total_interactions,unique_users,avg_dwell_time_ms
0,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,Apple Accessorie Stand,Electronics,6777,1557,17935.92
1,c1ab2206-9b50-4aaa-9446-f322d0976ea6,The Data Structures: A Comprehensive Guide,Books,3807,1088,18149.28
2,eaaf2b74-b09a-4bf5-b0b2-09021044cab8,Instant Pot Microfiber Kitchenware,Home & Kitchen,3359,1002,17108.04
3,58fd8edb-60ef-4856-bf1f-644ffc133d12,Wilson Book19 Fitness Equipment,Sports & Outdoors,3206,936,17785.97
4,1660a6f2-7ef5-4f47-a4aa-e917f41e014e,Tommy Hilfiger Womenswear Dress,Clothing & Accessories,2935,988,17735.21
5,2e1a3d22-2a05-4b6e-bfd8-4b5d5dab3009,Michelin Heavy-Duty Car Accessorie,Automotive,2599,829,17840.25
6,25a6c99c-d63c-4697-91ef-744b7879d605,Post-it Premium Stationery Set,Office Products,2586,638,18069.40
7,dbe68f3f-99a8-4ef2-a103-908a6fca244d,Wilson Fitness Equipment Pro,Sports & Outdoors,2488,831,17111.98
8,c2e39751-db17-424f-8d71-61ec7ecad424,Neutrogena Bath & Body Lotion,Beauty & Personal Care,2232,819,17328.76
9,d3701c21-709a-4090-bab2-020261ce8a92,Cuisinart Smart Appliance,Home & Kitchen,2113,788,18171.94
